# 01 · Qué hay en una grabación de teleoperación

Antes de convertir o entrenar nada, miramos el dato crudo tal como sale de la
teleoperación. La regla de la casa: **nadie entrena sobre datos que no miró**.
La mitad de los bugs de una política son bugs de datos.

Las grabaciones viven fuera del repo (`RAW_RECORDINGS_DIR`, por defecto
`~/Desktop/episodios`).

In [ ]:
# torch first: torchvision and torch._dynamo have a circular import that only
# resolves if torch is loaded before lerobot.
import torch  # noqa: F401
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from toolkit.paths import DATASETS_DIR, RAW_RECORDINGS_DIR

plt.rcParams["figure.dpi"] = 110

In [ ]:
TASK_DIR = RAW_RECORDINGS_DIR / "pick_mameluco"
episodes = sorted(p for p in TASK_DIR.iterdir() if p.is_dir())
print(len(episodes), "episodios:", episodes[0].name, "...", episodes[-1].name)

Cada episodio es una carpeta con un `data.json` (todo lo numérico) y las
imágenes de la cámara como JPEG sueltos. Abramos uno.

In [ ]:
episode_dir = TASK_DIR / "episode_0009"
raw = json.loads((episode_dir / "data.json").read_text())
print(raw.keys())
print("tarea:", raw["text"])
print("info.image:", raw["info"]["image"])

Ojo con esto: `info` dice que la imagen es de 480×640. Ahora miremos una imagen
de verdad.

In [ ]:
first_frame = sorted((episode_dir / "colors").glob("*.jpg"))[0]
image = cv2.imread(str(first_frame))
print("imagen real:", image.shape)

Es de **360×640**. La metadata miente. Por eso el conversor oficial de Unitree
(que asume 480×640) fallaba con nuestros datos, y por eso nuestro conversor lee
el tamaño de la imagen y no de la metadata. Lección general: medí el dato, no
confíes en lo que dice el encabezado.

Ahora, qué hay en cada cuadro:

In [ ]:
frame0 = raw["data"][0]
print("claves del cuadro:", list(frame0.keys()))
for part, values in frame0["states"].items():
    print(f"  states.{part:9s} -> {list(values.keys())}, qpos de {len(values['qpos'])} valores")
print("joint_names:", {k: len(v) for k, v in raw["info"]["joint_names"].items()})

Cada cuadro tiene dos cosas que importan:

- **`states`**: dónde *están* las articulaciones (lo que miden los encoders).
- **`actions`**: dónde el operador *pidió* que estén. Es lo que la política va a
  aprender a imitar.

Usamos brazos (7 + 7) y manos (6 + 6) = **26 números**. La cintura (`body`)
viene grabada pero no la usamos.

Para leerlo usamos el mismo lector que el conversor oficial, así lo que
miramos acá es exactamente lo que entra al dataset.

In [ ]:
from toolkit.dataset.convert_unitree import SingleTaskJsonDataset
from toolkit.dataset.robot_configs import G1_BRAINCO_1CAM, G1_BRAINCO_1CAM_CONFIG, register_robot_configs

register_robot_configs()
reader = SingleTaskJsonDataset(TASK_DIR, G1_BRAINCO_1CAM)  # lee los 65 data.json (tarda unos segundos)
JOINTS = G1_BRAINCO_1CAM_CONFIG.motors
index = reader.episode_paths.index(str(episode_dir))
item = reader.get_item(index)
state, action = item["state"], item["action"]
print("state", state.shape, "action", action.shape, "duración", round(len(state) / 30, 1), "s")

## Estado contra acción

Dibujamos el brazo derecho: en gris lo que el robot hizo (`state`), en color lo
que se le pidió (`action`).

In [ ]:
right_arm = [i for i, name in enumerate(JOINTS) if name.startswith("kRight")][:7]
t = np.arange(len(state)) / 30
fig, axes = plt.subplots(len(right_arm), 1, figsize=(10, 12), sharex=True)
for ax, j in zip(axes, right_arm):
    ax.plot(t, state[:, j], color="0.6", lw=2, label="state")
    ax.plot(t, action[:, j], lw=1, label="action")
    ax.set_ylabel(JOINTS[j].replace("kRight", ""), rotation=0, ha="right", fontsize=8)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("segundos")
plt.tight_layout()

Dos cosas para mirar:

1. La acción va **un poco adelante** del estado (mirá el codo cerca de los 10 s):
   el operador pide y el motor sigue con algo de retraso.
2. Cuando el brazo se queda quieto, a veces queda una **diferencia fija** entre
   lo pedido y lo alcanzado (la muñeca al final del episodio). Típicamente es la
   gravedad o la rigidez del controlador: el motor no llega del todo al objetivo.

Las dos son normales y son parte de lo que la política aprende. En el notebook
de validación las vamos a medir en serio, en todos los episodios.

## Las manos

Las manos Brainco tienen 6 grados de libertad cada una. Acá se ve el agarre.

In [ ]:
right_hand = [i for i, name in enumerate(JOINTS) if name.startswith("kRight")][7:]
fig, ax = plt.subplots(figsize=(10, 3))
for j in right_hand:
    ax.plot(t, action[:, j], label=JOINTS[j].replace("kRightHand", ""))
ax.set_xlabel("segundos"); ax.set_title("mano derecha (action)")
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()

## Qué ve la cámara a lo largo del episodio

In [ ]:
frames = item["cameras"]["cam_right_high"]
picks = np.linspace(0, len(frames) - 1, 5).astype(int)
fig, axes = plt.subplots(1, 5, figsize=(16, 2.6))
for ax, k in zip(axes, picks):
    ax.imshow(frames[k]); ax.set_title(f"t = {k / 30:.1f} s"); ax.axis("off")
plt.tight_layout()

## Duración de todos los episodios

Un primer chequeo de calidad barato: si un episodio dura mucho menos que el
resto, probablemente es un intento que se cortó.

In [ ]:
lengths = pd.Series({Path(p).name: len(d["data"]) for p, d in zip(reader.episode_paths, reader.episodes_data_cached)})
ax = (lengths / 30).plot.bar(figsize=(14, 3), color=["tab:red" if n < 150 else "tab:blue" for n in lengths])
ax.set_ylabel("segundos"); ax.set_xticks(range(0, len(lengths), 4)); ax.set_xticklabels(lengths.index[::4], rotation=90, fontsize=7)
plt.tight_layout()
lengths.sort_values().head(5)

Tres episodios duran menos de 4 segundos. En el notebook 03 vamos a ver que
quien entrenó la primera política los descartó.